# NEW

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import joblib
import os
from config.settings import get_path

# Cấu hình đường dẫn
DATA_DIR = "/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/preprocessed-dataset-news"
GT_FEB_PATH = get_path("/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/data/final_groundtruth.pkl")

# --- 1. Hàm Load User (Theo code bạn cung cấp) ---
def read_parquet_user(train_path: str):
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    user_chunk_files = [file for file in files if 'user_chunk' in file]
    print(f"Reading {len(user_chunk_files)} user chunk files...")
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
    return user_chunk_df

# --- 2. Load Transaction (Như bước trước) ---
def read_parquet_transaction(train_path: str):
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
    print(f"Reading {len(purchase_chunk_files)} transaction chunk files...")
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
    return purchase_chunk_df

# === THỰC HIỆN LOAD ===
print(">>> 1. Loading Data...")

# A. Transaction
pl_trans = read_parquet_transaction(DATA_DIR)
df_trans = pl_trans.to_pandas()
# Ép kiểu
if df_trans['created_date'].dtype == 'object':
    df_trans['created_date'] = pd.to_datetime(df_trans['created_date'])
df_trans['customer_id'] = df_trans['customer_id'].astype(str)
df_trans['item_id'] = df_trans['item_id'].astype(str)
print(f"   -> Transaction: {len(df_trans)} rows")

# B. User
pl_user = read_parquet_user(DATA_DIR)
df_user = pl_user.to_pandas()
df_user['customer_id'] = df_user['customer_id'].astype(str)

# Kiểm tra tên cột Province (thường là 'province' hoặc 'city')
# Ở đây giả sử là 'province', bạn đổi lại nếu khác
if 'province' not in df_user.columns and 'city' in df_user.columns:
    df_user.rename(columns={'city': 'province'}, inplace=True)

# Fill NA cho province
df_user['province'] = df_user['province'].fillna("UNKNOWN")

# Tạo Dictionary Mapping: User -> Province (để tra cứu cho nhanh)
user_province_map = df_user.set_index('customer_id')['province'].to_dict()

print(f"   -> User Info: {len(df_user)} rows. Mapped {len(user_province_map)} users.")
print(f"   -> Top Provinces: {df_user['province'].value_counts().head(5).to_dict()}")

# C. Ground Truth
gt_feb = joblib.load(GT_FEB_PATH)
print(f"   -> Feb GT: {len(gt_feb)} users")

>>> 1. Loading Data...
Reading 80 transaction chunk files...
   -> Transaction: 39028077 rows
Reading 10 user chunk files...
   -> User Info: 4573956 rows. Mapped 4573956 users.
   -> Top Provinces: {'Hồ Chí Minh': 1223328, 'Đồng Nai': 312131, 'Bình Dương': 287542, 'Hà Nội': 201491, 'Đà Nẵng': 146596}
   -> Feb GT: 644970 users


In [2]:
def get_cold_start_trends_by_province(df_trans, user_prov_map, hist_start, hist_end, target_start, target_end, top_k=10):
    """
    1. Tìm Cold Users trong tháng Target (những người không mua trong Hist).
    2. Map những Cold User này về Province của họ.
    3. Tính Top K item bán chạy nhất cho Cold User TẠI TỪNG PROVINCE.
    """
    print(f"\n[Trend Analysis] Period: {target_start} -> {target_end}")
    
    # 1. Filter Time
    mask_hist = (df_trans['created_date'] >= hist_start) & (df_trans['created_date'] <= hist_end)
    mask_target = (df_trans['created_date'] >= target_start) & (df_trans['created_date'] <= target_end)
    
    # Lấy tập user ID
    hist_users = set(df_trans.loc[mask_hist, 'customer_id'].unique())
    target_df = df_trans[mask_target].copy()
    target_users = set(target_df['customer_id'].unique())
    
    # 2. Identify Cold Users (Proxy)
    cold_users = target_users - hist_users
    print(f"   -> Found {len(cold_users)} proxy cold-start users.")
    
    if len(cold_users) == 0:
        return [], {}

    # 3. Filter Transaction của Cold Users
    df_cold_purchases = target_df[target_df['customer_id'].isin(cold_users)].copy()
    
    # Map Province vào Transaction
    df_cold_purchases['province'] = df_cold_purchases['customer_id'].map(user_prov_map).fillna("UNKNOWN")
    
    # --- A. Tính Global Trend (Dự phòng) ---
    global_top = df_cold_purchases['item_id'].value_counts().head(top_k).index.tolist()
    
    # --- B. Tính Province Trend ---
    province_trend_map = {}
    
    # Group theo Province và đếm Item
    # (Cách này nhanh hơn loop)
    grouped = df_cold_purchases.groupby(['province', 'item_id']).size().reset_index(name='count')
    
    # Sort count giảm dần
    grouped = grouped.sort_values(['province', 'count'], ascending=[True, False])
    
    # Lấy Top K cho mỗi group
    for province, group in grouped.groupby('province'):
        top_items = group.head(top_k)['item_id'].tolist()
        province_trend_map[province] = top_items
        
    print(f"   -> Generated trends for {len(province_trend_map)} provinces.")
    
    return global_top, province_trend_map

def evaluate_province_aware(gt_dict, history_users, user_prov_map, global_trend, province_trends, k=10):
    """
    Tính Precision cho User Cold Start thực sự.
    Logic:
      - Nếu User Cold Start có thông tin Province -> Dùng List của Province đó.
      - Nếu không có hoặc Province đó chưa có trend -> Dùng Global Trend.
    """
    precisions = []
    cold_count = 0
    used_province_trend = 0
    
    for user, true_items in gt_dict.items():
        user = str(user)
        # Chỉ tính Cold Start User
        if user not in history_users:
            cold_count += 1
            true_set = set([str(x) for x in true_items])
            
            # Lấy Province của User test
            user_prov = user_prov_map.get(user, "UNKNOWN")
            
            # Chọn danh sách gợi ý
            if user_prov in province_trends and len(province_trends[user_prov]) > 0:
                pred_items = province_trends[user_prov]
                used_province_trend += 1
            else:
                pred_items = global_trend # Fallback
                
            # Tính hits
            hits = len(set(pred_items[:k]) & true_set)
            precisions.append(hits / k)
            
    if cold_count == 0: return 0.0, 0, 0
    
    return np.mean(precisions), cold_count, used_province_trend

In [3]:
print("="*60)
print("🧪 SCENARIO 1: VALIDATION (JAN 2025) - PROVINCE AWARE")
print("="*60)

TOP_K = 10

# 1. Learn Trends from DEC 2024 (Grouped by Province)
# Cold Users T12 = Mua T12 nhưng ko mua T8-T11
global_trend_dec, prov_trends_dec = get_cold_start_trends_by_province(
    df_trans, user_province_map,
    hist_start='2024-08-01', hist_end='2024-11-30',
    target_start='2024-12-01', target_end='2024-12-31',
    top_k=TOP_K
)

print(f"   -> Global Trend Sample: {global_trend_dec[:3]}...")
if 'Ha Noi' in prov_trends_dec:
    print(f"   -> Ha Noi Trend Sample: {prov_trends_dec['Ha Noi'][:3]}...")

# 2. Evaluate on JAN 2025
print("\n>>> Evaluating on JAN 2025...")
# Tạo GT T1
df_jan = df_trans[(df_trans['created_date'] >= '2025-01-01') & (df_trans['created_date'] <= '2025-01-31')]
gt_jan = df_jan.groupby('customer_id')['item_id'].apply(list).to_dict()

# History check cho T1 là T8->T12
df_hist_jan = df_trans[(df_trans['created_date'] >= '2024-08-01') & (df_trans['created_date'] <= '2024-12-31')]
hist_users_jan = set(df_hist_jan['customer_id'].unique())

p10, n_cold, n_prov_used = evaluate_province_aware(
    gt_jan, hist_users_jan, user_province_map, 
    global_trend_dec, prov_trends_dec, k=TOP_K
)

print(f"\n📊 RESULT JAN 2025:")
print(f"   - Total Cold Users: {n_cold}")
print(f"   - Used Province Specific Trend: {n_prov_used} ({n_prov_used/n_cold*100:.1f}%)")
print(f"   - Precision@{TOP_K}: {p10:.6f}")

🧪 SCENARIO 1: VALIDATION (JAN 2025) - PROVINCE AWARE

[Trend Analysis] Period: 2024-12-01 -> 2024-12-31
   -> Found 174881 proxy cold-start users.
   -> Generated trends for 91 provinces.
   -> Global Trend Sample: ['4690000000001', '6766000000002', '1512000000004']...

>>> Evaluating on JAN 2025...

📊 RESULT JAN 2025:
   - Total Cold Users: 168669
   - Used Province Specific Trend: 168665 (100.0%)
   - Precision@10: 0.015397


In [4]:
print("\n" + "="*60)
print("🚀 SCENARIO 2: FINAL TEST (FEB 2025) - PROVINCE AWARE")
print("="*60)

# 1. Learn Trends from JAN 2025
# Cold Users T1 = Mua T1 nhưng ko mua T9-T12
global_trend_jan, prov_trends_jan = get_cold_start_trends_by_province(
    df_trans, user_province_map,
    hist_start='2024-09-01', hist_end='2024-12-31',
    target_start='2025-01-01', target_end='2025-01-31',
    top_k=TOP_K
)

print(f"   -> Global Trend Sample: {global_trend_jan[:3]}...")

# 2. Evaluate on FEB 2025 (Final GT)
print("\n>>> Evaluating on FEB 2025...")

# History check cho T2 là T9->T1
df_hist_feb = df_trans[(df_trans['created_date'] >= '2024-09-01') & (df_trans['created_date'] <= '2025-01-31')]
hist_users_feb = set(df_hist_feb['customer_id'].unique())

p10_feb, n_cold_feb, n_prov_used_feb = evaluate_province_aware(
    gt_feb, hist_users_feb, user_province_map, 
    global_trend_jan, prov_trends_jan, k=TOP_K
)

print(f"\n📊 RESULT FEB 2025:")
print(f"   - Total Cold Users: {n_cold_feb}")
print(f"   - Used Province Specific Trend: {n_prov_used_feb} ({n_prov_used_feb/n_cold_feb*100:.1f}%)")
print(f"   - Precision@{TOP_K}: {p10_feb:.6f}")


🚀 SCENARIO 2: FINAL TEST (FEB 2025) - PROVINCE AWARE

[Trend Analysis] Period: 2025-01-01 -> 2025-01-31
   -> Found 182662 proxy cold-start users.
   -> Generated trends for 81 provinces.
   -> Global Trend Sample: ['4690000000001', '1512000000004', '6766000000002']...

>>> Evaluating on FEB 2025...

📊 RESULT FEB 2025:
   - Total Cold Users: 2
   - Used Province Specific Trend: 0 (0.0%)
   - Precision@10: 0.000000


In [5]:
# --- 3. Hàm Load Item (Để lấy Category) ---
def read_parquet_item(train_path: str):
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    item_chunk_files = [file for file in files if 'item_chunk' in file]
    print(f"Reading {len(item_chunk_files)} item chunk files...")
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
    return item_chunk_df

# === THỰC HIỆN LOAD ITEM ===
print("\n>>> 3. Loading Item Info...")
pl_item = read_parquet_item(DATA_DIR)
df_item = pl_item.to_pandas()
df_item['item_id'] = df_item['item_id'].astype(str)

# Chọn các cột cần thiết để phân tích
# Ưu tiên category_l1/l2/l3 nếu có
cols_to_keep = ['item_id']
for c in ['category', 'category_l1', 'category_l2', 'category_l3']:
    if c in df_item.columns:
        cols_to_keep.append(c)
        df_item[c] = df_item[c].fillna('UNK') # Fill NA luôn

df_item_meta = df_item[cols_to_keep].drop_duplicates('item_id').set_index('item_id')
print(f"   -> Item Meta: {len(df_item_meta)} items loaded.")
print(f"   -> Columns: {cols_to_keep}")


>>> 3. Loading Item Info...
Reading 1 item chunk files...
   -> Item Meta: 27323 items loaded.
   -> Columns: ['item_id', 'category', 'category_l1', 'category_l2', 'category_l3']


In [6]:
# read csv
df_feb_new = pd.read_csv('submission_jan2025_new.csv')

df_feb_all = pd.read_csv('submission_jan2025_all.csv')

In [9]:
import pandas as pd
import numpy as np
import joblib
import os
from config.settings import get_path

# ... (Giữ nguyên phần import và load data ban đầu của bạn) ...

# ==============================================================================
# 1. HÀM TÍNH TREND (CẬP NHẬT: CHUẨN HÓA ITEM ID 13 KÝ TỰ)
# ==============================================================================
def get_cold_start_trends_by_province(df_trans, user_prov_map, hist_start, hist_end, target_start, target_end, top_k=10):
    print(f"\n[Trend Analysis] Period: {target_start} -> {target_end}")
    
    # Chuẩn hóa item_id trong transaction trước khi tính toán
    # Đảm bảo item nào cũng 13 ký tự
    df_trans['item_id'] = df_trans['item_id'].astype(str).str.zfill(13)

    # 1. Filter Time
    mask_hist = (df_trans['created_date'] >= hist_start) & (df_trans['created_date'] <= hist_end)
    mask_target = (df_trans['created_date'] >= target_start) & (df_trans['created_date'] <= target_end)
    
    hist_users = set(df_trans.loc[mask_hist, 'customer_id'].unique())
    target_df = df_trans[mask_target].copy()
    target_users = set(target_df['customer_id'].unique())
    
    # 2. Identify Cold Users
    cold_users = target_users - hist_users
    print(f"   -> Found {len(cold_users)} proxy cold-start users.")
    
    if len(cold_users) == 0:
        return [], {}

    # 3. Filter Transaction
    df_cold_purchases = target_df[target_df['customer_id'].isin(cold_users)].copy()
    df_cold_purchases['province'] = df_cold_purchases['customer_id'].map(user_prov_map).fillna("UNKNOWN")
    
    # --- A. Tính Global Trend ---
    global_top = df_cold_purchases['item_id'].value_counts().head(top_k).index.tolist()
    
    # --- B. Tính Province Trend ---
    province_trend_map = {}
    grouped = df_cold_purchases.groupby(['province', 'item_id']).size().reset_index(name='count')
    grouped = grouped.sort_values(['province', 'count'], ascending=[True, False])
    
    for province, group in grouped.groupby('province'):
        top_items = group.head(top_k)['item_id'].tolist()
        province_trend_map[province] = top_items
        
    print(f"   -> Generated trends for {len(province_trend_map)} provinces.")
    return global_top, province_trend_map

# ==============================================================================
# 2. HÀM REPLACE VÀ FORMAT ITEM ID (CẬP NHẬT)
# ==============================================================================
def apply_cold_start_replacement(df_sub, hist_users, user_map, global_trend, prov_trends):
    """
    Thay thế item của Cold User và format item_id thành 13 ký tự.
    """
    print(f"\n[Processing Submission] Input shape: {df_sub.shape}")
    
    # Chuẩn hóa item_id của file input trước (cho chắc)
    df_sub['item_id'] = df_sub['item_id'].astype(str).str.zfill(13)
    
    # 1. Tách Cold và Warm User
    all_sub_users = df_sub['customer_id'].unique()
    cold_users = set(all_sub_users) - hist_users
    
    print(f"   -> Found {len(cold_users)} Cold Users need replacement.")
    
    # Nếu không có Cold User, vẫn phải trả về df đã format item_id
    if len(cold_users) == 0:
        return df_sub
    
    # 2. Giữ lại data của Warm Users
    df_warm = df_sub[~df_sub['customer_id'].isin(cold_users)].copy()
    
    # 3. Tạo data mới cho Cold Users 
    new_rows = []
    
    for uid in cold_users:
        prov = user_map.get(str(uid), "UNKNOWN")
        
        # Chọn list items
        if prov in prov_trends and len(prov_trends[prov]) > 0:
            items = prov_trends[prov]
            if len(items) < 10:
                fill = [x for x in global_trend if x not in items]
                items = items + fill[:10-len(items)]
        else:
            items = global_trend
            
        items = items[:10]
        
        for rank, item_id in enumerate(items):
            # Format item_id ngay lúc tạo dòng mới
            item_id_formatted = str(item_id).zfill(13)
            
            new_rows.append({
                'customer_id': uid,
                'item_id': item_id_formatted,
                'pred_score': 0.99 - (rank * 0.01) 
            })
            
    df_cold_new = pd.DataFrame(new_rows)
    
    # 4. Gộp lại
    df_final = pd.concat([df_warm, df_cold_new], ignore_index=True)
    
    # --- BƯỚC QUAN TRỌNG: CHUẨN HÓA LẦN CUỐI CHO TOÀN BỘ DATA ---
    # Đảm bảo cả Warm User (cũ) và Cold User (mới) đều có item_id 13 ký tự
    df_final['item_id'] = df_final['item_id'].astype(str).str.zfill(13)
    
    # Sort lại
    df_final = df_final.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    
    print(f"   -> Done. Output shape: {df_final.shape}")
    print(f"   -> Sample Item ID: {df_final['item_id'].iloc[0]} (Len: {len(df_final['item_id'].iloc[0])})")
    
    return df_final

# ==============================================================================
# 3. CHẠY THỰC TẾ
# ==============================================================================

# A. Tính Trend (Học từ tháng 1)
print("\n>>> Calculating Trends from Jan 2025...")
global_trend_jan, prov_trends_jan = get_cold_start_trends_by_province(
    df_trans, user_province_map,
    hist_start='2024-09-01', hist_end='2024-12-31',
    target_start='2025-01-01', target_end='2025-01-31',
    top_k=10
)

# B. Xác định History Users (Warm Users - đã có trước tháng 2)
# Lưu ý: Vì ta predict cho tháng 2, nên Warm User là người có history <= 31/01/2025
df_history_all = df_trans[df_trans['created_date'] <= '2025-01-31']
history_users_set = set(df_history_all['customer_id'].unique())
print(f"   -> Identified {len(history_users_set)} Warm Users.")

# C. Xử lý File Submission
print("\n>>> Reading & Processing Submission Files...")
try:
    df_feb_new = pd.read_csv('submission_jan2025_new.csv')
    df_feb_all = pd.read_csv('submission_jan2025_all.csv')
    
    # Đảm bảo customer_id là string
    df_feb_new['customer_id'] = df_feb_new['customer_id'].astype(str)
    df_feb_all['customer_id'] = df_feb_all['customer_id'].astype(str)
    
    # --- Xử lý NEW ---
    print("\n>>> Processing NEW ITEM Submission...")
    df_feb_new_fixed = apply_cold_start_replacement(
        df_feb_new, history_users_set, user_province_map, 
        global_trend_jan, prov_trends_jan
    )
    df_feb_new_fixed.to_csv('submission_jan2025_new_coldfixed.csv', index=False)
    print("✅ Saved: submission_jan2025_new_coldfixed.csv")

    # --- Xử lý ALL ---
    print("\n>>> Processing ALL ITEM Submission...")
    df_feb_all_fixed = apply_cold_start_replacement(
        df_feb_all, history_users_set, user_province_map, 
        global_trend_jan, prov_trends_jan
    )
    df_feb_all_fixed.to_csv('submission_jan2025_all_coldfixed.csv', index=False)
    print("✅ Saved: submission_jan2025_all_coldfixed.csv")

except FileNotFoundError:
    print("❌ Error: Could not find submission csv files. Check paths.")
except Exception as e:
    print(f"❌ Error: {e}")


>>> Calculating Trends from Jan 2025...

[Trend Analysis] Period: 2025-01-01 -> 2025-01-31
   -> Found 184052 proxy cold-start users.
   -> Generated trends for 63 provinces.
   -> Identified 2569978 Warm Users.

>>> Reading & Processing Submission Files...

>>> Processing NEW ITEM Submission...

[Processing Submission] Input shape: (6449700, 3)
   -> Found 135235 Cold Users need replacement.
   -> Done. Output shape: (6449700, 3)
   -> Sample Item ID: 4690000000001 (Len: 13)
✅ Saved: submission_jan2025_new_coldfixed.csv

>>> Processing ALL ITEM Submission...

[Processing Submission] Input shape: (6449700, 3)
   -> Found 135235 Cold Users need replacement.
   -> Done. Output shape: (6449700, 3)
   -> Sample Item ID: 4690000000001 (Len: 13)
✅ Saved: submission_jan2025_all_coldfixed.csv


In [1]:
import pandas as pd
import json
import os

def process_csv_to_json(csv_path, json_output_path):
    print(f"\n>>> Processing: {csv_path}")
    
    if not os.path.exists(csv_path):
        print(f"❌ File not found: {csv_path}")
        return

    # 1. Đọc CSV với dtype=str để giữ nguyên số 0 ở đầu
    # Đây là bước quan trọng nhất để không bị mất format
    df = pd.read_csv(
        csv_path, 
        dtype={'customer_id': str, 'item_id': str, 'pred_score': float}
    )
    
    # 2. Chuẩn hóa lại cho chắc chắn (Safety check)
    # Đảm bảo item_id đủ 13 ký tự (thêm số 0 vào đầu nếu thiếu)
    df['item_id'] = df['item_id'].str.zfill(13)
    
    # 3. Sắp xếp dữ liệu
    # Group theo User, và trong mỗi User sắp xếp Item theo điểm số GIẢM DẦN
    # Để item có khả năng mua cao nhất nằm đầu list
    df = df.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    
    # 4. Group by và tạo Dictionary
    # Kết quả: {'user_id': ['item_id_1', 'item_id_2', ...]}
    # Cột pred_score sẽ tự động bị loại bỏ trong bước này, chỉ lấy item_id
    final_dict = df.groupby('customer_id')['item_id'].apply(list).to_dict()
    
    # In kiểm tra mẫu
    first_key = list(final_dict.keys())[0]
    print(f"   -> Sample User: {first_key}")
    print(f"   -> Top Items: {final_dict[first_key][:3]}...")
    print(f"   -> Total Users: {len(final_dict)}")

    # 5. Lưu ra file JSON
    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(final_dict, f, ensure_ascii=False)
        
    print(f"✅ Saved JSON to: {json_output_path}")

# =========================================================
# THỰC THI
# =========================================================

# 1. Xử lý file NEW
process_csv_to_json(
    csv_path='submission_jan2025_new_coldfixed.csv',
    json_output_path='submission_feb2025_new_coldfixed.json'
)

# 2. Xử lý file ALL
process_csv_to_json(
    csv_path='submission_jan2025_all_coldfixed.csv',
    json_output_path='submission_feb2025_all_coldfixed.json'
)


>>> Processing: submission_jan2025_new_coldfixed.csv
   -> Sample User: 1000011
   -> Top Items: ['4690000000001', '1512000000004', '2024000000010']...
   -> Total Users: 644970
✅ Saved JSON to: submission_feb2025_new_coldfixed.json

>>> Processing: submission_jan2025_all_coldfixed.csv
   -> Sample User: 1000011
   -> Top Items: ['4690000000001', '1512000000004', '0007090000158']...
   -> Total Users: 644970
✅ Saved JSON to: submission_feb2025_all_coldfixed.json


In [6]:
import pandas as pd
import numpy as np
import pickle
import json
import os
from config.settings import get_path

# ==============================================================================
# 1. HÀM CHUYỂN ĐỔI PICKLE DICT -> DATAFRAME
# ==============================================================================
def load_pred_pkl_to_df(pkl_path):
    """
    Đọc file .pkl (Dict hoặc DataFrame) và chuẩn hóa về DataFrame 
    có 3 cột: [customer_id, item_id, pred_score]
    """
    print(f"\n📂 Loading Prediction from: {pkl_path}")
    if not os.path.exists(pkl_path):
        print(f"❌ File not found: {pkl_path}")
        return pd.DataFrame()

    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    # TRƯỜNG HỢP 1: Dữ liệu là Dictionary {user: [item1, item2...]}
    if isinstance(data, dict):
        print(f"   -> Detected Dictionary format. Converting to DataFrame...")
        rows = []
        for user, items in data.items():
            # Chỉ lấy Top 20 để xử lý cho nhẹ (nếu list quá dài)
            top_items = items[:20] 
            for rank, item in enumerate(top_items):
                rows.append({
                    "customer_id": str(user),
                    "item_id": str(item).zfill(13), # Fix số 0 ngay lúc load
                    # Tạo score giả lập để giữ thứ tự: Rank 0 -> 1.0, Rank 1 -> 0.99...
                    "pred_score": 1.0 - (rank * 0.001) 
                })
        df = pd.DataFrame(rows)

    # TRƯỜNG HỢP 2: Dữ liệu đã là DataFrame
    elif isinstance(data, pd.DataFrame):
        print(f"   -> Detected DataFrame format.")
        df = data.copy()
        # Đảm bảo có cột pred_score, nếu không thì tạo đại
        if 'pred_score' not in df.columns:
            df['pred_score'] = 1.0 
        
        # Chuẩn hóa ID
        df['customer_id'] = df['customer_id'].astype(str)
        df['item_id'] = df['item_id'].astype(str).str.zfill(13)

    else:
        print("❌ Unknown data format!")
        return pd.DataFrame()

    print(f"   -> Loaded {len(df)} rows. Sample Item: {df['item_id'].iloc[0]}")
    return df

# ==============================================================================
# 2. HÀM TÍNH TREND (GIỮ NGUYÊN LOGIC CŨ)
# ==============================================================================
def get_cold_start_trends_by_province(df_trans, user_prov_map, hist_start, hist_end, target_start, target_end, top_k=10):
    # ... (Giữ nguyên logic của bạn ở trên) ...
    # Để code ngắn gọn, tôi giả định bạn đã có hàm này hoặc dùng lại hàm cũ.
    # Logic quan trọng nhất là df_trans['item_id'] = df_trans['item_id'].astype(str).str.zfill(13)
    
    print(f"\n[Trend Analysis] Period: {target_start} -> {target_end}")
    df_trans['item_id'] = df_trans['item_id'].astype(str).str.zfill(13) # Chuẩn hóa item

    mask_target = (df_trans['created_date'] >= target_start) & (df_trans['created_date'] <= target_end)
    mask_hist = (df_trans['created_date'] >= hist_start) & (df_trans['created_date'] <= hist_end)
    
    hist_users = set(df_trans.loc[mask_hist, 'customer_id'].unique())
    target_df = df_trans[mask_target].copy()
    target_users = set(target_df['customer_id'].unique())
    
    cold_users = target_users - hist_users
    print(f"   -> Found {len(cold_users)} proxy cold-start users.")
    
    if len(cold_users) == 0: return [], {}

    df_cold_purchases = target_df[target_df['customer_id'].isin(cold_users)].copy()
    df_cold_purchases['province'] = df_cold_purchases['customer_id'].map(user_prov_map).fillna("UNKNOWN")
    
    global_top = df_cold_purchases['item_id'].value_counts().head(top_k).index.tolist()
    
    province_trend_map = {}
    grouped = df_cold_purchases.groupby(['province', 'item_id']).size().reset_index(name='count')
    grouped = grouped.sort_values(['province', 'count'], ascending=[True, False])
    
    for province, group in grouped.groupby('province'):
        province_trend_map[province] = group.head(top_k)['item_id'].tolist()
        
    return global_top, province_trend_map

# ==============================================================================
# 3. HÀM REPLACE COLD START (GIỮ NGUYÊN LOGIC)
# ==============================================================================
def apply_cold_start_replacement(df_sub, hist_users, user_map, global_trend, prov_trends):
    # df_sub lúc này đã là DataFrame chuẩn từ hàm load_pred_pkl_to_df
    
    all_sub_users = df_sub['customer_id'].unique()
    cold_users = set(all_sub_users) - hist_users
    print(f"   -> Found {len(cold_users)} Cold Users need replacement.")
    
    if len(cold_users) == 0: return df_sub
    
    df_warm = df_sub[~df_sub['customer_id'].isin(cold_users)].copy()
    new_rows = []
    
    for uid in cold_users:
        prov = user_map.get(str(uid), "UNKNOWN")
        if prov in prov_trends and len(prov_trends[prov]) > 0:
            items = prov_trends[prov]
            if len(items) < 10:
                fill = [x for x in global_trend if x not in items]
                items = items + fill[:10-len(items)]
        else:
            items = global_trend
            
        items = items[:10]
        for rank, item_id in enumerate(items):
            new_rows.append({
                'customer_id': str(uid),
                'item_id': str(item_id).zfill(13),
                'pred_score': 0.99 - (rank * 0.01)
            })
            
    df_cold_new = pd.DataFrame(new_rows)
    df_final = pd.concat([df_warm, df_cold_new], ignore_index=True)
    df_final = df_final.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    
    return df_final

# ==============================================================================
# 4. HÀM EXPORT JSON (TỪ DATAFRAME TRONG RAM)
# ==============================================================================
def export_df_to_json(df, json_output_path):
    print(f"   -> Exporting to JSON: {json_output_path}")
    
    # 1. Đảm bảo format
    df['customer_id'] = df['customer_id'].astype(str)
    df['item_id'] = df['item_id'].astype(str).str.zfill(13)
    
    # 2. Sort lại lần cuối
    df = df.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    
    # 3. Group by -> Dict
    final_dict = df.groupby('customer_id')['item_id'].apply(list).to_dict()
    
    # 4. Save
    with open(json_output_path, 'w', encoding='utf-8') as f:
        json.dump(final_dict, f, ensure_ascii=False)
    print(f"✅ Saved JSON with {len(final_dict)} users.")

# ==============================================================================
# CHẠY THỰC TẾ
# ==============================================================================

# [SETUP DATA] Giả sử bạn đã load df_trans và user_province_map như cũ
# df_trans = ... 
# user_province_map = ...

# A. Tính Trend (Ví dụ)
print("\n>>> 1. Calculating Trends...")
global_trend_jan, prov_trends_jan = get_cold_start_trends_by_province(
    df_trans, user_province_map,
    hist_start='2024-09-01', hist_end='2024-12-31',
    target_start='2025-01-01', target_end='2025-01-31'
)

# B. Xác định Warm Users
df_history_all = df_trans[df_trans['created_date'] <= '2025-01-31']
history_users_set = set(df_history_all['customer_id'].astype(str).unique())

# C. Xử lý File PKL và xuất JSON
# Cấu hình đường dẫn file PKL đầu vào
PATH_PKL_NEW = get_path("/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/old-artifacts/artifacts/predictions_new_item_rec.pkl")
PATH_PKL_ALL = get_path("/datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/old-artifacts/artifacts/predictions_all_item_rec.pkl")


>>> 1. Calculating Trends...

[Trend Analysis] Period: 2025-01-01 -> 2025-01-31
   -> Found 182662 proxy cold-start users.


In [7]:
import pickle

def export_df_to_pkl(df, pkl_output_path):
    print(f"   -> Exporting to PKL: {pkl_output_path}")
    
    # 1. Đảm bảo format string và 13 ký tự
    df['customer_id'] = df['customer_id'].astype(str)
    df['item_id'] = df['item_id'].astype(str).str.zfill(13)
    
    # 2. Sort lại lần cuối (User tăng dần, Score giảm dần để item tốt nhất lên đầu)
    df = df.sort_values(['customer_id', 'pred_score'], ascending=[True, False])
    
    # 3. Group by -> Dict {user: [list_items]}
    final_dict = df.groupby('customer_id')['item_id'].apply(list).to_dict()
    
    # 4. Save pickle
    with open(pkl_output_path, 'wb') as f:
        pickle.dump(final_dict, f)
        
    print(f"✅ Saved PKL with {len(final_dict)} users to {pkl_output_path}")

In [8]:
# --- Xử lý NEW ---
print("\n>>> 2. Processing NEW ITEM (from PKL)...")
df_new = load_pred_pkl_to_df(PATH_PKL_NEW) 

if not df_new.empty:
    # Áp dụng logic thay thế cold-start
    df_new_fixed = apply_cold_start_replacement(
        df_new, history_users_set, user_province_map, 
        global_trend_jan, prov_trends_jan
    )
    
    # [THAY ĐỔI] Lưu xuống file PKL
    export_df_to_pkl(df_new_fixed, 'submission_feb2025_new_coldfixed.pkl')

# --- Xử lý ALL ---
print("\n>>> 3. Processing ALL ITEM (from PKL)...")
df_all = load_pred_pkl_to_df(PATH_PKL_ALL)

if not df_all.empty:
    # Áp dụng logic thay thế cold-start
    df_all_fixed = apply_cold_start_replacement(
        df_all, history_users_set, user_province_map, 
        global_trend_jan, prov_trends_jan
    )
    
    # [THAY ĐỔI] Lưu xuống file PKL
    export_df_to_pkl(df_all_fixed, 'submission_feb2025_all_coldfixed.pkl')


>>> 2. Processing NEW ITEM (from PKL)...

📂 Loading Prediction from: /datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/old-artifacts/artifacts/predictions_new_item_rec.pkl
   -> Detected Dictionary format. Converting to DataFrame...
   -> Loaded 12899400 rows. Sample Item: 7218000000002
   -> Found 135235 Cold Users need replacement.
   -> Exporting to PKL: submission_feb2025_new_coldfixed.pkl
✅ Saved PKL with 644970 users to submission_feb2025_new_coldfixed.pkl

>>> 3. Processing ALL ITEM (from PKL)...

📂 Loading Prediction from: /datastore/uittogether/LuuTru/Thanhld/CS116-DoAn/Phase-2/Pipeline-training3/Thanh/old-artifacts/artifacts/predictions_all_item_rec.pkl
   -> Detected Dictionary format. Converting to DataFrame...
   -> Loaded 12899400 rows. Sample Item: 2803000000013
   -> Found 135235 Cold Users need replacement.
   -> Exporting to PKL: submission_feb2025_all_coldfixed.pkl
✅ Saved PKL with 644970 users to submission_feb2025_all_coldfixed.pkl
